In [1]:
#Notebook 4 — Baseline XGBoost, where we establish the initial XGBoost 
# benchmark before SHAP-guided feature pruning and before Optuna tuning.
#That ordering is important because it gives you a legitimate baseline against 
# which the later S-XGBoost / SP-XGBoost improvements can be evaluated.

# ============================================================
# CELL 1--NOTEBOOK 4 — BASELINE XGBOOST
# ============================================================

import pandas as pd
import numpy as np
import xgboost as xgb

from pathlib import Path

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("=" * 70)
print("NOTEBOOK 4 — BASELINE XGBOOST")
print("=" * 70)

print(f"XGBoost version: {xgb.__version__}")

NOTEBOOK 4 — BASELINE XGBOOST
XGBoost version: 3.3.0


In [2]:
#CELL -2
# Project directories
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
FIELD_DIR = DATA_DIR / "field"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURE_DIR = RESULTS_DIR / "figures"
TABLE_DIR = RESULTS_DIR / "tables"
REPORT_DIR = RESULTS_DIR / "reports"
# Create output directories
for folder in [FIGURE_DIR, TABLE_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


print(f"Processed data directory:\n{PROCESSED_DIR}")

Processed data directory:
..\data\processed


In [3]:
# ============================================================
# CELL 3- LOAD BINARY MODELLING DATA
# ============================================================

train_binary_model = pd.read_csv(
    PROCESSED_DIR / "train_binary_model.csv"
)

test_binary_model = pd.read_csv(
    PROCESSED_DIR / "test_binary_model.csv"
)

print("=" * 70)
print("BINARY MODELLING DATA LOADED")
print("=" * 70)

print(
    f"Training: {train_binary_model.shape}"
)

print(
    f"Testing:  {test_binary_model.shape}"
)

BINARY MODELLING DATA LOADED
Training: (468, 62)
Testing:  (117, 62)


In [4]:
# ============================================================
# C3LL 4-- SEPARATE FEATURES AND TARGET(X & Y)
# ============================================================

TARGET = "RISK_BINARY"

X_train = train_binary_model.drop(
    columns=[TARGET]
)

y_train = train_binary_model[TARGET].copy()

X_test = test_binary_model.drop(
    columns=[TARGET]
)

y_test = test_binary_model[TARGET].copy()


print("=" * 70)
print("FEATURE / TARGET SEPARATION")
print("=" * 70)

print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")

print(f"X_test:  {X_test.shape}")
print(f"y_test:  {y_test.shape}")

FEATURE / TARGET SEPARATION
X_train: (468, 61)
y_train: (468,)
X_test:  (117, 61)
y_test:  (117,)


In [5]:
#Before training the model, validate the exact modelling matrix.
# ============================================================
# BASELINE DATA VALIDATION
# ============================================================

print("=" * 70)
print("BASELINE DATA VALIDATION")
print("=" * 70)

assert X_train.shape == (468, 61)
assert X_test.shape == (117, 61)

assert y_train.shape == (468,)
assert y_test.shape == (117,)

assert X_train.isnull().sum().sum() == 0
assert X_test.isnull().sum().sum() == 0

assert X_train.select_dtypes(
    exclude=np.number
).shape[1] == 0

assert X_test.select_dtypes(
    exclude=np.number
).shape[1] == 0

assert list(X_train.columns) == list(X_test.columns)

assert sorted(y_train.unique()) == [0, 1]
assert sorted(y_test.unique()) == [0, 1]

print("✓ Training shape: 468 × 61")
print("✓ Testing shape: 117 × 61")
print("✓ No missing values")
print("✓ All predictors numeric")
print("✓ Train/test feature alignment confirmed")
print("✓ Binary target coding confirmed")
print("\n✓ BASELINE DATA VALIDATION PASSED")

BASELINE DATA VALIDATION
✓ Training shape: 468 × 61
✓ Testing shape: 117 × 61
✓ No missing values
✓ All predictors numeric
✓ Train/test feature alignment confirmed
✓ Binary target coding confirmed

✓ BASELINE DATA VALIDATION PASSED


In [6]:
#Cell 6 — Lock the baseline configuration
#This is important for reproducibility.
#We are deliberately not tuning these parameters yet.
# ============================================================
# BASELINE XGBOOST CONFIGURATION
# ============================================================

BASELINE_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": 42,
    "n_estimators": 100,
    "max_depth": 6,
    "learning_rate": 0.3,
    "subsample": 1.0,
    "colsample_bytree": 1.0
}

print("=" * 70)
print("BASELINE XGBOOST CONFIGURATION")
print("=" * 70)

for parameter, value in BASELINE_PARAMS.items():
    print(f"{parameter:20s}: {value}")

print("\n✓ Default-style baseline configuration locked")
print("✓ No SHAP feature pruning")
print("✓ No Optuna tuning")
print("✓ No resampling")
print("✓ No class weighting")

BASELINE XGBOOST CONFIGURATION
objective           : binary:logistic
eval_metric         : logloss
random_state        : 42
n_estimators        : 100
max_depth           : 6
learning_rate       : 0.3
subsample           : 1.0
colsample_bytree    : 1.0

✓ Default-style baseline configuration locked
✓ No SHAP feature pruning
✓ No Optuna tuning
✓ No resampling
✓ No class weighting


In [7]:
## ============================================================
# Cell - 7 TRAIN BASELINE XGBOOST
# ============================================================

baseline_xgb = xgb.XGBClassifier(
    **BASELINE_PARAMS
)

baseline_xgb.fit(
    X_train,
    y_train
)

print("=" * 70)
print("BASELINE XGBOOST TRAINING COMPLETE")
print("=" * 70)

print("✓ Model trained on 468 training observations")
print("✓ 61 predictors used")
print("✓ No synthetic observations")
print("✓ No test data used during training")

BASELINE XGBOOST TRAINING COMPLETE
✓ Model trained on 468 training observations
✓ 61 predictors used
✓ No synthetic observations
✓ No test data used during training


In [8]:
# ============================================================
# CELL 8 --BASELINE PREDICTIONS ---Generate predictions
# ============================================================

y_pred = baseline_xgb.predict(X_test)

y_prob = baseline_xgb.predict_proba(
    X_test
)[:, 1]

print("=" * 70)
print("BASELINE PREDICTIONS GENERATED")
print("=" * 70)

print(f"Predicted classes: {len(y_pred)}")
print(f"Predicted probabilities: {len(y_prob)}")

assert len(y_pred) == 117
assert len(y_prob) == 117

print("✓ Prediction count matches test set")

BASELINE PREDICTIONS GENERATED
Predicted classes: 117
Predicted probabilities: 117
✓ Prediction count matches test set


In [9]:
#Cell 9 — Calculate primary baseline metrics
# ============================================================
# BASELINE PERFORMANCE METRICS
# ============================================================

roc_auc = roc_auc_score(
    y_test,
    y_prob
)

pr_auc = average_precision_score(
    y_test,
    y_prob
)

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

cm = confusion_matrix(
    y_test,
    y_pred
)

tn, fp, fn, tp = cm.ravel()

specificity = tn / (tn + fp)


print("=" * 70)
print("BASELINE XGBOOST PERFORMANCE")
print("=" * 70)

print(f"ROC-AUC:       {roc_auc:.4f}")
print(f"PR-AUC:        {pr_auc:.4f}")
print(f"F1-score:      {f1:.4f}")
print(f"Precision:     {precision:.4f}")
print(f"Recall:        {recall:.4f}")
print(f"Specificity:   {specificity:.4f}")
print(f"Accuracy:      {accuracy:.4f}")

print("\nConfusion Matrix:")
print(cm)

BASELINE XGBOOST PERFORMANCE
ROC-AUC:       0.7567
PR-AUC:        0.7220
F1-score:      0.6296
Precision:     0.6538
Recall:        0.6071
Specificity:   0.7049
Accuracy:      0.6581

Confusion Matrix:
[[43 18]
 [22 34]]


In [10]:
#Cell 10 — Classification report
# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Not-At-Risk",
            "At-Risk"
        ],
        zero_division=0
    )
)

CLASSIFICATION REPORT
              precision    recall  f1-score   support

 Not-At-Risk       0.66      0.70      0.68        61
     At-Risk       0.65      0.61      0.63        56

    accuracy                           0.66       117
   macro avg       0.66      0.66      0.66       117
weighted avg       0.66      0.66      0.66       117



In [11]:
#Cell 11 — Save baseline results
# ============================================================
# SAVE BASELINE RESULTS
# ============================================================
RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

baseline_results = pd.DataFrame({
    "Model": ["Baseline XGBoost"],
    "ROC_AUC": [roc_auc],
    "PR_AUC": [pr_auc],
    "F1": [f1],
    "Precision": [precision],
    "Recall": [recall],
    "Specificity": [specificity],
    "Accuracy": [accuracy],
    "TN": [tn],
    "FP": [fp],
    "FN": [fn],
    "TP": [tp]
})

baseline_results.to_csv(
    RESULTS_DIR / "baseline_xgboost_binary_results.csv",
    index=False
)

print("=" * 70)
print("BASELINE RESULTS SAVED")
print("=" * 70)

print(
    RESULTS_DIR /
    "baseline_xgboost_binary_results.csv"
)

print("\n✓ Baseline result recorded")

BASELINE RESULTS SAVED
..\results\baseline_xgboost_binary_results.csv

✓ Baseline result recorded


In [12]:
#Cell 12 — Final Notebook 4 validation
# ============================================================
# NOTEBOOK 4 — FINAL VALIDATION
# ============================================================

print("=" * 70)
print("NOTEBOOK 4 — FINAL VALIDATION")
print("=" * 70)

assert X_train.shape == (468, 61)
assert X_test.shape == (117, 61)

assert len(y_pred) == 117
assert len(y_prob) == 117

assert np.all(
    (y_prob >= 0) &
    (y_prob <= 1)
)

assert np.isfinite(roc_auc)
assert np.isfinite(pr_auc)
assert np.isfinite(f1)
assert np.isfinite(precision)
assert np.isfinite(recall)
assert np.isfinite(specificity)
assert np.isfinite(accuracy)

assert cm.shape == (2, 2)

print("✓ Training data: 468 × 61")
print("✓ Test data: 117 × 61")
print("✓ Predictions: 117")
print("✓ Probabilities valid")
print("✓ All evaluation metrics finite")
print("✓ Confusion matrix valid")
print("✓ Test set remains 117 observations")

print("\n" + "=" * 70)
print("✓ NOTEBOOK 4 BASELINE VALIDATION PASSED")
print("=" * 70)

NOTEBOOK 4 — FINAL VALIDATION
✓ Training data: 468 × 61
✓ Test data: 117 × 61
✓ Predictions: 117
✓ Probabilities valid
✓ All evaluation metrics finite
✓ Confusion matrix valid
✓ Test set remains 117 observations

✓ NOTEBOOK 4 BASELINE VALIDATION PASSED


In [13]:
# ============================================================
# CELL 13 — SAVE BASELINE XGBOOST MODEL
# ============================================================

import joblib

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

baseline_model_path = (
    MODELS_DIR / "baseline_xgboost_binary.pkl"
)

joblib.dump(
    baseline_xgb,
    baseline_model_path
)

print("=" * 70)
print("BASELINE XGBOOST MODEL SAVED")
print("=" * 70)

print(f"Model: {baseline_model_path}")

# Verify that the file exists
assert baseline_model_path.exists()

print("✓ Model file exists")
print("✓ Baseline XGBoost model successfully saved")

BASELINE XGBOOST MODEL SAVED
Model: ..\models\baseline_xgboost_binary.pkl
✓ Model file exists
✓ Baseline XGBoost model successfully saved


In [14]:
# ============================================================
# CELL 14 — VERIFY SAVED BASELINE MODEL
#Verifying that the saved model can actually be reloaded.
# ============================================================

loaded_baseline_xgb = joblib.load(
    baseline_model_path
)

# Generate predictions from the reloaded model
loaded_pred = loaded_baseline_xgb.predict(X_test)
loaded_prob = loaded_baseline_xgb.predict_proba(X_test)[:, 1]

# Confirm predictions are identical
assert np.array_equal(
    y_pred,
    loaded_pred
)

assert np.allclose(
    y_prob,
    loaded_prob
)

print("=" * 70)
print("BASELINE MODEL RELOAD VALIDATION")
print("=" * 70)

print("✓ Model successfully reloaded")
print("✓ Class predictions identical")
print("✓ Probability predictions identical")
print("✓ Saved model integrity confirmed")

print("\n✓ BASELINE XGBOOST MODEL VERIFIED")

BASELINE MODEL RELOAD VALIDATION
✓ Model successfully reloaded
✓ Class predictions identical
✓ Probability predictions identical
✓ Saved model integrity confirmed

✓ BASELINE XGBOOST MODEL VERIFIED
